# Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import axes


# Load data

In [ ]:
games_odds = pd.read_csv("data/game_odds.csv")


In [ ]:
games_index = pd.read_csv("data/games_index.csv")

In [ ]:
games_schedule = pd.read_csv("data/games_schedule.csv")

In [ ]:
play_by_play = pd.read_csv("data/play_by_play.csv")

In [ ]:
player_boxscores = pd.read_csv("data/player_boxscores.csv")

In [ ]:
team_boxscores = pd.read_csv("data/team_boxscores.csv")

# Clean data

In [ ]:
#get rid of nan values in the datasets
games_odds = games_odds.dropna()
games_index = games_index.dropna()
games_schedule = games_schedule.dropna()

#play_by_play = play_by_play.dropna() # don't drop because it'll drop every row that's not a shoot, because the subtype column is "Unknown" for every action that's not a shoot, and we need to keep those actions to be able to analyze the best players
#insted : drop rows where player_name_i is nan
play_by_play = play_by_play.dropna(subset=['player_name_i'])

player_boxscores = player_boxscores.dropna()
team_boxscores = team_boxscores.dropna()

# See data

In [ ]:
#get all action types ranked by number of occurences in the play_by_play dataset
action_types_count = play_by_play['action_type'].value_counts().reset_index()
action_types_count.columns = ['action_type', 'count']
print("Action types ranked by number of occurences:")
display(action_types_count)

In [ ]:
play_by_play_columns = ['game_id', 'game_date', 'game_matchup', 'team_id', 'person_id',
       'action_id', 'action_number', 'clock', 'period', 'team_tri_code',
       'player_name', 'player_name_i', 'xlegacy', 'ylegacy', 'shot_distance',
       'shot_result', 'is_field_goal', 'score_home', 'score_away',
       'points_total', 'location', 'description', 'action_type', 'subtype',
       'video_available', 'shot_value']

# Players analysis

## Player infos (full name, abreviate name, team abreviation, season year) in one place : get by functions

In [ ]:
# new column little_name : LeBron James -> L. James in player_boxscores dataset
player_boxscores["little_name"] = player_boxscores["player_name"].apply(lambda x: x[0] + ". " + x.split(" ")[-1])

# aggregate player_boxscores to get the player_id, player_name, little_name, team_abbreviation and season_year for each player_id
player_simple_infos = (
    player_boxscores
    .groupby(
        ["season_year", "team_abbreviation", "player_id"],
        as_index=False
    )
    .agg({
        "player_name": "first",
        "little_name": "first",
    })
)

#-display(player_simple_infos.head())

def get_player_full_name(player_id):
    player_info = player_simple_infos[player_simple_infos["player_id"] == player_id]
    if not player_info.empty:
        return player_info["player_name"].values[0]
    else:
        return None
def get_player_little_name(player_id):
    player_info = player_simple_infos[player_simple_infos["player_id"] == player_id]
    if not player_info.empty:
        return player_info["little_name"].values[0]
    else:
        return None

#display all the rows where the little_name is L. James
l_james_rows = player_simple_infos[player_simple_infos["little_name"] == "L. James"]
print("Rows where little_name is L. James:")
display(l_james_rows)

## Player_boxscores

### preparation : dict for column names + aggregate

In [ ]:
stat_columns_player_boxscores = {
        "pts":   "points",
        "reb":   "rebounds",
        "ast":   "assists",
        "stl":   "steals",
        "blk":   "blocks",
        "tov":   "turnovers",
        "min":   "minutes",
        "fgm":   "Field Goals Made",
        "fga":   "Field Goals Attempted",
        "fg3m":  "Three Pointers Made",
        "fg3a":  "Three Pointers Attempted",
        "ftm":   "Free Throws Made",
        "fta":   "Free Throws Attempted",
        "plus_minus": "impact of player on team performance",
        "dreb":  "defensive rebounds",
        "oreb":  "offensive rebounds",
    }
available = {k: v for k, v in stat_columns_player_boxscores.items() if k in player_boxscores.columns}
player_wins_per_team = (
        player_boxscores.groupby(["player_id", "player_name", "team_abbreviation"], as_index=False)
          .agg(
              games=("game_id", "nunique"),
              **{v: (k, "mean") for k, v in available.items()}
          )
    )
# print("Aggregated Player Stats:")
# display(player_wins_per_team)

### Player_boxscores get top_players_points_avg, top_player_rebounds, top_player_assists

In [ ]:
# Plotting average points per game for top 20 players
top_players_points_avg = player_wins_per_team.sort_values(by="points", ascending=False).head(20).copy()
top_players_points_avg["little_name"] = top_players_points_avg["player_id"].apply(get_player_little_name)

#print(top_players_points_avg[["player_name", "points", "team_abbreviation"]])

top_player_rebounds_avg = player_wins_per_team.sort_values(by="rebounds", ascending=False).head(20).copy()
top_player_assists_avg = player_wins_per_team.sort_values(by="assists", ascending=False).head(20).copy()

# add little_name column without relying on index labels
top_player_rebounds_avg["little_name"] = top_player_rebounds_avg["player_id"].apply(get_player_little_name)
top_player_assists_avg["little_name"] = top_player_assists_avg["player_id"].apply(get_player_little_name)


### Plot data : avg points per game, avg points + avg rebounds + avg assists per game

In [ ]:
# Plotting average points per game for top 10 players without taking into account the team they played for
plt.figure(figsize=(12, 6))
plt.bar(top_players_points_avg["player_name"] + " (" + top_players_points_avg["team_abbreviation"] + ")", top_players_points_avg["points"])
plt.xlabel("Player Name")
plt.ylabel("Average Points per Game")
plt.title("Top 20 Players by Average Points per Game")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()


# plot better players by points, rebounds and assists
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].bar(top_players_points_avg["player_name"], top_players_points_avg["points"], color='blue')
axes[0].set_xlabel("Player Name")
axes[0].set_ylabel("Average Points per Game")
axes[0].set_title("Top 20 Players by Average Points per Game")
axes[0].tick_params(axis='x', rotation=75)
axes[1].bar(top_player_rebounds_avg["player_name"], top_player_rebounds_avg["rebounds"], color='orange')
axes[1].set_xlabel("Player Name")
axes[1].set_ylabel("Average Rebounds per Game")
axes[1].set_title("Top 20 Players by Average Rebounds per Game")
axes[1].tick_params(axis='x', rotation=75)
axes[2].bar(top_player_assists_avg["player_name"], top_player_assists_avg["assists"], color='green')
axes[2].set_xlabel("Player Name")
axes[2].set_ylabel("Average Assists per Game")
axes[2].set_title("Top 20 Players by Average Assists per Game")
axes[2].tick_params(axis='x', rotation=75)
plt.tight_layout()
plt.show()



### Player by team and season

In [ ]:
# # print player name, team abreviation, season from player_boxscores dataset
summary_player_by_team_by_player = (
    player_boxscores.groupby(["season_year", "team_abbreviation", "player_id"], as_index=False)
    .agg({
        "player_name": "first",
    })
)
summary_player_by_team_by_player["little_name"] = summary_player_by_team_by_player["player_id"].apply(get_player_little_name)
    
display(summary_player_by_team_by_player.head())

### Best Shooter (points + nb of shoots made) (shoot made = panier marqué)

In [ ]:
all_shoots_made = play_by_play[play_by_play['shot_result'] == 'Made']

# all points scored by each player in the dataset
player_points_total = all_shoots_made.groupby(["player_name_i", "person_id"], as_index=False)["shot_value"].sum().reset_index()
player_points_total = player_points_total.sort_values(by="shot_value", ascending=False)


#plotting the top 20 players by total points scored in the dataset
plt.figure(figsize=(10, 4))
plt.bar(player_points_total["player_name_i"].head(20), player_points_total["shot_value"].head(20))
plt.xlabel("Player Name")
plt.ylabel("Total Points Scored")
plt.title("Top 20 Players by Total Points Scored")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

# nb of shoot made by each player in the dataset
player_shoot_made = all_shoots_made.groupby(["player_name_i", "person_id"], as_index=False)["shot_value"].count().reset_index()
player_shoot_made = player_shoot_made.sort_values(by="shot_value", ascending=False)


plt.figure(figsize=(10, 4))
plt.bar(player_shoot_made["player_name_i"].head(20), player_shoot_made["shot_value"].head(20))
plt.xlabel("Player Name")
plt.ylabel("Total Shoot Made")
plt.title("Top 20 Players by Total Shoot Made")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()




In [ ]:
# check if a player is in the player_shoot_made and top_players_points_avg
best_shooter_players = set(player_shoot_made["player_name_i"].head(20)).intersection(set(top_players_points_avg["little_name"].head(20)))
print("Players in both top 20 lists (aka best shooters):", best_shooter_players)

### Best rebound player

In [ ]:
#get all rebound rows
all_rebounds_made = play_by_play[play_by_play['action_type'] == 'Rebound']

#count all the rebounds subtypes (defensive, offensive, unknown) and rank them by number of occurences
rebound_subtypes_count = all_rebounds_made['subtype'].value_counts().reset_index()
rebound_subtypes_count.columns = ['subtype', 'count']


# summ all the rebounds made by each player in the dataset
player_rebounds_total = all_rebounds_made.groupby(["player_name_i", "person_id"])["action_id"].count().reset_index()
player_rebounds_total = player_rebounds_total.sort_values(by="action_id", ascending=False).copy()

# add little name column without relying on index labels
player_rebounds_total["little_name"] = player_rebounds_total["person_id"].apply(get_player_little_name)


#subplot of top_player_rebounds_avg and player_rebounds_total head(20)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].bar(top_player_rebounds_avg["player_name"], top_player_rebounds_avg["rebounds"], color='orange')
axes[0].set_xlabel("Player Name")
axes[0].set_ylabel("Average Rebounds per Game")
axes[0].set_title("Top 20 Players by Average Rebounds per Game")
axes[0].tick_params(axis='x', rotation=75)
axes[1].bar(player_rebounds_total["player_name_i"].head(20), player_rebounds_total["action_id"].head(20), color='orange')
axes[1].set_xlabel("Player Name")
axes[1].set_ylabel("Total Rebounds Made")
axes[1].set_title("Top 20 Players by Total Rebounds Made")
axes[1].tick_params(axis='x', rotation=75)
plt.tight_layout()
plt.show()


#get all players that are in top_player_rebounds_avg and player_rebounds_total
best_rebounder_players = set(player_rebounds_total["little_name"].head(20)).intersection(set(top_player_rebounds_avg["little_name"].head(20)))
print("Players in both top 20 lists (aka best rebounders):", best_rebounder_players)


### Best Assist Player

In [ ]:
# get best assist player
best_assist_player_global = player_boxscores.sort_values(by="ast", ascending=False)
# keep only the top 20 players by assists without double player_id
best_assist_player_global = best_assist_player_global.drop_duplicates(subset=["player_id"]).head(20).copy()

best_assist_player_global["little_name"] = best_assist_player_global["player_id"].apply(get_player_little_name)


#subplot of top_player_assists_avg and player_assists_total head(20)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].bar(top_player_assists_avg["little_name"], top_player_assists_avg["assists"], color='green')
axes[0].set_xlabel("Player Name")
axes[0].set_ylabel("Average Assists per Game")
axes[0].set_title("Top 20 Players by Average Assists per Game")
axes[0].tick_params(axis='x', rotation=75)
axes[1].bar(best_assist_player_global["little_name"], best_assist_player_global["ast"], color='green')
axes[1].set_xlabel("Player Name")
axes[1].set_ylabel("Total Assists Made")
axes[1].set_title("Top 20 Players by Total Assists Made")
axes[1].tick_params(axis='x', rotation=75)
plt.tight_layout()
plt.show()
    

#get all players that are in top_player_assists_avg and player_assists_total
best_assister_players = set(best_assist_player_global["little_name"].head(20)).intersection(set(top_player_assists_avg["little_name"].head(20)))
print("Players in both top 20 lists (aka best assisters):", best_assister_players)


## Best players

In [ ]:
best_players = []

for player in best_shooter_players:
    best_players.append(player)
for player in best_rebounder_players:
    best_players.append(player)
for player in best_assister_players:
    best_players.append(player)

#clean duplicates in best_players
best_players = list(set(best_players))
display(best_players)

In [ ]:
dict_best_players__player_id= {
    player_boxscores[player_boxscores["little_name"] == player]["player_id"].values[0]: player
    for player in best_players
}

best_players_full_name = [get_player_full_name(player_id) for player_id in dict_best_players__player_id.keys()]
display(best_players_full_name)

# Star Players (with scoring)

In [ ]:
# Aggregate player stats
player_summary = (
    player_boxscores
    .groupby(['player_id','player_name'], as_index=False)
    .agg(
        games=('game_id','nunique'),
        points=('pts','mean'),
        rebounds=('reb','mean'),
        assists=('ast','mean'),
        steals=('stl','mean'),
        blocks=('blk','mean'),
        fgm=('fgm','mean'), # field goals made
        fga=('fga','mean'), # field goals attempted
        fg3m=('fg3m','mean'), # three-point field goals made
        fg3a=('fg3a','mean'), # three-point field goals attempted
        ftm=('ftm','mean'), # free throws made
        fta=('fta','mean') # free throws attempted
    )
)

player_summary['field_goal_ratio'] = player_summary['fgm'] / player_summary['fga'] # field goal percentage
player_summary['three_point_ratio'] = player_summary['fg3m'] / player_summary['fg3a'] # three-point field goal percentage
player_summary['true_shooting_ratio'] = player_summary['points'] / (2*(player_summary['fga'] + 0.44*player_summary['fta'])) # true shooting percentage

player_summary.head()

In [ ]:
# Star score
player_summary['star_score'] = (
    player_summary['points']*0.40 +
    player_summary['rebounds']*0.20 +
    player_summary['assists']*0.20 +
    player_summary['steals']*0.10 +
    player_summary['blocks']*0.10
)

top_stars = player_summary.sort_values('star_score', ascending=False).head(20)
top_stars[['player_name','star_score']]

# list with the top 20 players by star score
top_20_players_star_score = top_stars['player_name'].tolist()
display(top_20_players_star_score)

In [ ]:
# plot the top 20 players by star score
plt.figure(figsize=(10, 6))
plt.bar(top_stars['player_name'], top_stars['star_score'])
plt.xlabel('Player Name')
plt.ylabel('Star Score')
plt.title('Top 20 Players by Star Score')
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

# display 3 best rebounders, assisters and shooters that are in the top_stars 
top_rebounders = top_stars.sort_values('rebounds', ascending=False).head(3)
top_assisters = top_stars.sort_values('assists', ascending=False).head(3)
top_shooters = top_stars.sort_values('points', ascending=False).head(3)
print("Top 3 Rebounders:")
display(top_rebounders[['player_name', 'rebounds']])
print("Top 3 Assisters:")
display(top_assisters[['player_name', 'assists']])

# plot the top 3 rebounders, assisters and shooters
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].bar(top_rebounders['player_name'], top_rebounders['rebounds'], color='orange')
axes[0].set_xlabel("Player Name")
axes[0].set_ylabel("Average Rebounds per Game")
axes[0].set_title("Top 3 Rebounders")
axes[0].tick_params(axis='x', rotation=75)
axes[1].bar(top_assisters['player_name'], top_assisters['assists'], color='orange')
axes[1].set_xlabel("Player Name")
axes[1].set_ylabel("Average Assists per Game")
axes[1].set_title("Top 3 Assisters")
axes[1].tick_params(axis='x', rotation=75)
axes[2].bar(top_shooters['player_name'], top_shooters['points'], color='orange')
axes[2].set_xlabel("Player Name")
axes[2].set_ylabel("Average Points per Game")
axes[2].set_title("Top 3 Shooters")
axes[2].tick_params(axis='x', rotation=75)
plt.tight_layout()
plt.show()


# Teams analysis

## Preparation

### Dictionary teams name - abreviation

In [ ]:
teams_dict = {
    "ATL": "Atlanta Hawks",
    "BOS": "Boston Celtics",
    "BKN": "Brooklyn Nets",
    "CHA": "Charlotte Hornets",
    "CHI": "Chicago Bulls",
    "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors",
    "HOU": "Houston Rockets",
    "IND": "Indiana Pacers",
    "LAC": "Los Angeles Clippers",
    "LAL": "Los Angeles Lakers",
    "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "NOP": "New Orleans Pelicans",
    "NYK": "New York Knicks",
    "OKC": "Oklahoma City Thunder",
    "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers",
    "PHX": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "SAC": "Sacramento Kings",
    "SAS": "San Antonio Spurs",
    "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz",
    "WAS": "Washington Wizards"
}

name_to_abbreviation = {v: k for k, v in teams_dict.items()}

# add LA Clippers as LAC and Charlotte Bobcats as CHA in name_to_abbreviation, because they are in the team_boxscores dataset
# but not in the teams_dict (don't add in teams_dict because it's not a real team, it's a former team that doesn't exist anymore, but it exists in the team_boxscores dataset)
name_to_abbreviation["Los Angeles Clippers"] = "LAC"
name_to_abbreviation["Charlotte Bobcats"] = "CHA"


#print(team_boxscores['team_name'].unique())

In [ ]:
#game_index : add team_abbreviation_home, team_abbreviation_away from teams_dict
games_index["team_abbreviation_home"] = games_index["team_name_home"].map(name_to_abbreviation)
games_index["team_abbreviation_away"] = games_index["team_name_away"].map(name_to_abbreviation)

### Aggregations from team_boxscores

In [ ]:
#teams aggregate stats from team boxscores
team_agg_summary = (
        team_boxscores.groupby(["team_id", "team_abbreviation"], as_index=False)
            .agg(
                games=("game_id", "nunique"),
                **{v: (k, "mean") for k, v in available.items()}
            )
    )
print("Aggregated Team Stats:")
display(team_agg_summary.head())

print()
# teams aggregate stats per season from team boxscores
team_agg_summary_per_season = (
        team_boxscores.groupby(["team_id", "team_abbreviation", "season_year"], as_index=False)
            .agg(
                games=("game_id", "nunique"),
                **{v: (k, "mean") for k, v in available.items()}
            )
    )
print("Aggregated Team Stats per Season:")
display(team_agg_summary_per_season)

In [ ]:
team_agg_summary_per_season['shoot_made_ratio'] = team_agg_summary_per_season['Field Goals Made'] / team_agg_summary_per_season['Field Goals Attempted']
team_agg_summary_per_season['three_point_made_ratio'] = team_agg_summary_per_season['Three Pointers Made'] / team_agg_summary_per_season['Three Pointers Attempted']
team_agg_summary_per_season.head(20)

display(team_agg_summary_per_season[team_agg_summary_per_season['team_abbreviation'] == 'LAL'])

# plot the evolution of the shoot made ratio and three point made ratio for the LAL team over the seasons
lal_data = team_agg_summary_per_season[team_agg_summary_per_season['team_abbreviation'] == 'LAL']
plt.figure(figsize=(12, 6))
plt.plot(lal_data['season_year'], lal_data['shoot_made_ratio'], label='Shoot Made Ratio')
plt.plot(lal_data['season_year'], lal_data['three_point_made_ratio'], label='Three Point Made Ratio')
plt.xlabel('Season Year')
plt.ylabel('Ratio')
plt.title('Evolution of Shoot Made Ratio and Three Point Made Ratio for LAKERS')
plt.legend()
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()


### Aggregation from games_index

In [ ]:
#teams aggregate stats per season from games_index
# "game_id","team_name_home", "team_name_away","winner", "season_year"
team_agg_summary_per_season_games_index = (
        games_index.groupby(["game_id","team_name_home", "team_name_away","winner", "season_year"], as_index=False)
            .agg(
                games=("game_id", "nunique"),
            )
    )
print("Aggregated Team Stats per Season from games_index:")
display(team_agg_summary_per_season_games_index)

In [ ]:
#count nb games per season from games_index
games_per_season = games_index.groupby("season_year")["game_id"].nunique().reset_index().sort_values(by="season_year", ascending=True)
print("Number of games per season from games_index:")
#display(games_per_season)

# count nb games per season per team from games_index
games_per_season_per_team = games_index.groupby(["season_year", "team_name_home"])["game_id"].nunique().reset_index().sort_values(by=["season_year", "team_name_home"], ascending=True) 
display(games_per_season_per_team)

In [ ]:
# get all the wins for each team from games_index 
team_wins = games_index["winner"].value_counts()
# print("Team Wins:")
# display(team_wins)

# bar plot of team wins per season year
team_wins_per_year = games_index.groupby(["season_year", "winner"]).size().unstack(fill_value=0)
team_wins_per_year.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.xlabel("Season Year")
plt.ylabel("Number of Wins")
plt.title("Team Wins per Season Year")
plt.legend(title="Team", bbox_to_anchor=(1, 1), loc='upper left')
plt.tight_layout()
plt.show()

#aggregate team wins per season year and team abbreviation
team_wins_per_season_per_team = games_index.groupby(["season_year", "winner"]).size().reset_index(name="wins")

display(team_wins_per_year.head())

print()

display(team_wins_per_season_per_team.head())

# list of the winning team per season year
winning_teams_per_season = team_wins_per_season_per_team.groupby("season_year")["winner"].apply(list).reset_index(name="winning_teams")
print("Winning teams per season year:")
display(winning_teams_per_season)

# list of the team with more wins per season year
most_wins_per_season = team_wins_per_season_per_team.loc[team_wins_per_season_per_team.groupby("season_year")["wins"].idxmax()].reset_index(drop=True)
print("Teams with most wins per season year:")
display(most_wins_per_season)


In [ ]:
# is a top_star player in the team with most wins per season year that year ? (check if the player name is in the winning team name that season year)
# Example: Check if 'LeBron James' is a top_star player in the team with most wins per season year that year

# for each top_star player aggregate season_year, team_abreviation, player_name from player_boxscores dataset 
top_stars_info = (
    player_boxscores.groupby(["season_year", "team_abbreviation", "player_id", "little_name"], as_index=False)
    .agg({
        "player_name": "first",
    })
)

display(top_stars_info.head())

# check if a top_star player is in the team with most wins per season year that year
top_stars_in_winning_teams = []
for index, row in most_wins_per_season.iterrows():
    season_year = row["season_year"]
    winning_team = row["winner"]
    top_stars_in_season = top_stars_info[top_stars_info["season_year"] == season_year]
    top_stars_in_winning_team = top_stars_in_season[top_stars_in_season["team_abbreviation"] == winning_team]
    top_stars_in_winning_teams.append(top_stars_in_winning_team)
print("Top star players in the teams with most wins per season year:")
display(top_stars_in_winning_teams) 

# Visualisation of data (use of course info)

In [ ]:
#GeoJson of the teams in the dataset in usa map
# dictionary with team abreviation (for all teams in team_dict) as key and the corresponding geojson as value
team_abbreviations = teams_dict.keys()

teams_locations = {
    "ATL": [33.755, -84.39],
    "BOS": [42.3601, -71.0589],
    "BKN": [40.6812, -73.9712],
    "CHA": [35.2271, -80.8431],
    "CHI": [41.8781, -87.6298],
    "CLE": [41.4967, -81.6855],
    "DAL": [32.7765, -96.7970],
    "DEN": [39.7392, -104.9903],
    "DET": [42.3314, -83.0458],
    "GSW": [37.7849, -122.4194],
    "HOU": [29.7604, -95.3698],
    "IND": [39.7682, -86.1583],
    "LAC": [34.0522, -118.2437],
    "LAL": [34.0522, -118.2437],
    "MEM": [35.1495, -89.9660],
    "MIA": [25.7617, -80.1918],
    "MIL": [43.0389, -87.9065],
    "MIN": [44.9792, -93.2638],
    "NOP": [29.9572, -90.0765],
    "NYK": [40.7128, -74.0060],
    "OKC": [35.4676, -97.5164],
    "ORL": [28.5383, -81.3792],
    "PHI": [39.9526, -75.1652],
    "PHX": [33.4484, -112.0740],
    "POR": [45.5122, -122.6683],
    "SAC": [38.5767, -121.4937],
    "SAS": [29.4267, -98.4313],
    "TOR": [43.6426, -79.3871],
    "UTA": [40.7608, -111.8910],
    "WAS": [38.8951, -77.0364]
}


usa_teams_geojson_dict_for_plot = {
    team: {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [location[1], location[0]]  # GeoJSON uses [longitude, latitude]
        },
        "properties": {
            "team": team,
            "team_name": teams_dict[team]
        }
    }
    for team, location in teams_locations.items()
}

#plot geojson of the teams in the dataset in usa map with plotly
import plotly.express as px
fig = px.scatter_geo(
    lat=[location[0] for location in teams_locations.values()],
    lon=[location[1] for location in teams_locations.values()],
    text=[teams_dict[team] for team in teams_locations.keys()],
    hover_name=[teams_dict[team] for team in teams_locations.keys()],
    title="NBA Teams Locations",
    scope="usa"
)
fig.update_traces(marker=dict(size=10, color='red'))
fig.show()

In [ ]:
# get last team of each top_stars from summary_player_by_team_by_player
top_stars_with_team = pd.merge(top_stars, summary_player_by_team_by_player, on="player_name", how="left")
top_stars_with_team = top_stars_with_team.sort_values(by="season_year", ascending=False).drop_duplicates(subset=["player_name"], keep="first")
top_stars_with_team[["player_name", "team_abbreviation", "season_year", "star_score"]]

nb_of_top_stars_per_team = top_stars_with_team['team_abbreviation'].value_counts().reset_index()
nb_of_top_stars_per_team.columns = ['team_abbreviation', 'count']
#display(nb_of_top_stars_per_team)

nb_of_top_stars_per_team['location'] = nb_of_top_stars_per_team['team_abbreviation'].map(teams_locations)


# add state name to nb_of_top_stars_per_team for choropleth map
state_abbreviations = {
    "ATL": "GA",
    "BOS": "MA",
    "BKN": "NY",
    "CHA": "NC",
    "CHI": "IL",
    "CLE": "OH",
    "DAL": "TX",
    "DEN": "CO",
    "DET": "MI",
    "GSW": "CA",
    "HOU": "TX",
    "IND": "IN",
    "LAC": "CA",
    "LAL": "CA",
    "MEM": "TN",
    "MIA": "FL",
    "MIL": "WI",
    "MIN": "MN",
    "NOP": "LA",
    "NYK": "NY",
    "OKC": "OK",
    "ORL": "FL",
    "PHI": "PA",
    "PHX": "AZ",
    "POR": "OR",
    "SAC": "CA",
    "SAS": "TX",
    "TOR": "ON", # Ontario, not a US state
    "UTA": "UT",
    "WAS": "DC" # District of Columbia, not a US state
}
nb_of_top_stars_per_team['state'] = nb_of_top_stars_per_team['team_abbreviation'].map(state_abbreviations)

display(nb_of_top_stars_per_team)

#show how many players per state
players_per_state = nb_of_top_stars_per_team.groupby('state')['count'].sum().reset_index()
players_per_state.columns = ['state', 'player_count']
display(players_per_state)



In [ ]:
# plot a choropleth map to see nb_of_top_stars_per_team in the USA
import plotly.express as px
fig = px.scatter_geo(
    lat=nb_of_top_stars_per_team['location'].apply(lambda x: x[0]),
    lon=nb_of_top_stars_per_team['location'].apply(lambda x: x[1]),
    text=nb_of_top_stars_per_team['team_abbreviation'] + ": " + nb_of_top_stars_per_team['count'].astype(str),
    hover_name=nb_of_top_stars_per_team['team_abbreviation'],
    title="Number of Top Stars per Team",
    scope="usa"
)
fig.update_traces(marker=dict(size=5, color='blue'), textposition="top center")
fig.update_layout(geo=dict(lakecolor='rgb(255, 255, 255)'),
                  width=1200,   
                  height=600)
fig.show()

fig = px.choropleth(
    players_per_state,
    locations='state',
    locationmode='USA-states',
    color='player_count',
    scope="usa",
    title="Number of Top Stars per State"
)
fig.update_layout(geo=dict(lakecolor='rgb(255, 255, 255)'),
                  width=1200,   
                  height=600)

import plotly.graph_objects as go

# Add state labels with counts
for _, row in players_per_state.iterrows():
    fig.add_trace(go.Scattergeo(
        locations=[row['state']],
        locationmode='USA-states',
        text=f"{row['state']} {row['player_count']}",
        mode='text',
        textfont=dict(size=10, color='red', family='Arial Black'),
        showlegend=False
    ))

fig.show()

In [ ]:
# add state column to most_wins_per_season
most_wins_per_season['state'] = most_wins_per_season['winner'].map(state_abbreviations)
display(most_wins_per_season)

fig = px.choropleth(
    most_wins_per_season,
    locations='state',
    locationmode='USA-states',
    color='wins',
    scope="usa",
    animation_frame='season_year',  # ← slider to browse seasons
    title="State with Most Wins per Season",
    color_continuous_scale='Viridis'
)

fig.update_layout(
    geo=dict(lakecolor='rgb(255, 255, 255)'),
    width=1200,
    height=600
)

fig.show()

fig = px.scatter_geo(
    most_wins_per_season,
    locations='state',
    locationmode='USA-states',
    size='wins',                    # ← dot size proportional to wins
    color='wins',                   # ← optional: also color by wins
    text='winner',
    hover_name='winner',
    animation_frame='season_year',  # ← optional: slider by season
    title="Team Wins per Season",
    scope="usa",
    size_max=50                     # ← max dot size, adjust as needed
)

fig.update_traces(textposition='top center')
fig.update_layout(
    geo=dict(lakecolor='rgb(255, 255, 255)'),
    width=1200,
    height=600
)

fig.show()



In [ ]:
# Count how many seasons each team was the winner
win_counts = most_wins_per_season.groupby('winner').size().reset_index(name='times_best')
win_counts['location'] = win_counts['winner'].map(teams_locations)

fig = px.scatter_geo(
    win_counts,
    lat=win_counts['location'].apply(lambda x: x[0]),
    lon=win_counts['location'].apply(lambda x: x[1]),
    size='times_best',
    color='times_best',
    text='winner',
    hover_name='winner',
    title="Times That a Team Had The Most Wins In The Season (1996-2026)",
    scope="usa",
    size_max=50
)

fig.update_traces(textposition='top center')
fig.update_layout(
    geo=dict(lakecolor='rgb(255, 255, 255)'),
    width=1200,
    height=600
)

fig.show()

In [ ]:
display(top_stars)

# Lakers

In [ ]:
# lakers stats per season from team_agg_summary_per_season
lakers_stats_per_season = team_agg_summary_per_season[team_agg_summary_per_season['team_abbreviation'] == 'LAL'].sort_values(by='season_year', ascending=True)
display(lakers_stats_per_season)

In [ ]:
# analyze the evolution of the lakers stats per season and compare it to the star score of the top star players in the lakers team per season (from top_stars_with_team)
lakers_top_stars = top_stars_with_team[top_stars_with_team['team_abbreviation'] == 'LAL'].sort_values(by='season_year', ascending=True)
display(lakers_top_stars)

In [ ]:
# plot lakers stats per season
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
axes[0].plot(lakers_stats_per_season['season_year'], lakers_stats_per_season['points'], marker='o')
axes[0].set_title('Lakers Average Points per Game per Season')
axes[0].set_xlabel('Season Year')
axes[0].set_ylabel('Average Points per Game')
axes[0].tick_params(axis='x', rotation=75)
axes[1].plot(lakers_stats_per_season['season_year'], lakers_stats_per_season['assists'], marker='o', color='green')
axes[1].set_title('Lakers Average Assists per Game per Season')
axes[1].set_xlabel('Season Year')
axes[1].set_ylabel('Average Assists per Game')
axes[1].tick_params(axis='x', rotation=75)
axes[2].plot(lakers_stats_per_season['season_year'], lakers_stats_per_season['rebounds'], marker='o', color='orange')
axes[2].set_title('Lakers Average Rebounds per Game per Season')
axes[2].set_xlabel('Season Year')
axes[2].set_ylabel('Average Rebounds per Game')
axes[2].tick_params(axis='x', rotation=75)
plt.tight_layout()
plt.show()